# 详细文档: 水文模型 - 产流模块

**相关模块:** `hydro_model/runoff.py`

## 目的
此文档旨在详细介绍`hydro_model`中的两种产流模块，并对它们进行对比。产流模块是水文模型的核心之一，它负责计算在给定降雨和蒸散发条件下，有多少降雨能够转化为地表径流。

此笔记本将：
1.  详细解释`SimpleRunoffModule`（一个基于蓄满产流概念的简单模型）的原理和参数。
2.  详细解释`SCSCurveNumberModule`（经典的SCS-CN产流模型）的原理和参数。
3.  在相同的降雨输入下，运行这两个模块，并比较它们的产流结果。
4.  通过可视化来直观地展示两种方法的差异。

## 1. 环境设置

首先，我们导入所需的库和我们自己的产流模块。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.runoff import SimpleRunoffModule, SCSCurveNumberModule

## 2. 简单产流模块 (SimpleRunoffModule)

这是一个概念性的、基于“蓄水池”隐喻的产流模型。它的核心思想是：
- 流域内有一个土壤水“蓄水池”，其最大容量为 `S_max`。
- 当降雨时，一部分雨水会填满这个蓄水池，另一部分则根据当前土壤的饱和程度（`S / S_max`）产流。
- 土壤越湿，同样降雨产生的径流就越多。
- 每天还会有一定的土壤水通过蒸发（`pet`）和损失（`c_loss`）而减少。

**优点**: 能够模拟流域干湿程度对产流的影响，考虑了前期影响雨量（通过`S`的状态）。
**缺点**: 参数（特别是`c_loss`）物理意义不明确，需要率定。

In [ ]:
# 定义参数并实例化
simple_runoff = SimpleRunoffModule(S_max=200, c_loss=0.05)

# 定义一个降雨序列
sample_rainfall = np.array([0, 5, 10, 30, 50, 20, 10, 5, 0, 0, 15, 35, 10, 0])
sample_pet = np.full_like(sample_rainfall, 1.0) # 假设每天蒸散发1mm

# 运行模拟
simple_runoff_results = []
for i in range(len(sample_rainfall)):
    runoff = simple_runoff.run(rainfall=sample_rainfall[i], pet=sample_pet[i])
    simple_runoff_results.append(runoff)

print("SimpleRunoffModule 产流结果 (mm):")
print(np.round(simple_runoff_results, 2))

## 3. SCS-CN 产流模块 (SCSCurveNumberModule)

这是水文学中广泛使用的产流计算方法。它的核心思想是：
- 流域的产流潜力由一个单一的、综合性的参数`CN`（Curve Number，径流曲线数）来表征。`CN`值越大，产流能力越强。
- 降雨开始时，一部分雨量（称为初损 `Ia`）会被截留和入渗，不产生径流。
- 超过初损后，降雨才开始根据`CN`值推算出的一个公式来划分径流和入渗。

**优点**: 理论成熟，`CN`值可以根据土地利用、土壤类型和前期湿润条件从手册中查到，物理意义相对明确。
**缺点**: 原始的SCS-CN模型不直接考虑时间过程，它计算的是一场降雨的总径流量。在这个实现中，通过在每个时间步独立调用它，来近似模拟连续的产流过程，但这忽略了状态的连续性（即前一天的湿润程度）。

In [ ]:
# 定义参数并实例化 (CN=85 是一个典型值)
scs_runoff = SCSCurveNumberModule(CN=85)

# 使用相同的降雨序列运行模拟
scs_runoff_results = []
for i in range(len(sample_rainfall)):
    runoff = scs_runoff.run(rainfall=sample_rainfall[i])
    scs_runoff_results.append(runoff)

print("SCSCurveNumberModule 产流结果 (mm):")
print(np.round(scs_runoff_results, 2))

## 4. 结果对比与可视化

现在我们将两个模块的产流结果与原始的降雨过程绘制在一起，进行对比。

In [ ]:
timesteps = np.arange(len(sample_rainfall))

fig, ax = plt.subplots(figsize=(15, 7))

# 绘制降雨 (作为参考)
ax.bar(timesteps, sample_rainfall, color='c', alpha=0.5, label='Rainfall')

# 绘制产流结果
ax.plot(timesteps, simple_runoff_results, 'b-o', label='SimpleRunoffModule')
ax.plot(timesteps, scs_runoff_results, 'r-s', label='SCSCurveNumberModule (CN=85)')

ax.set_title('Comparison of Runoff Modules')
ax.set_xlabel('Time Step (days)')
ax.set_ylabel('Depth (mm)')
ax.legend()
ax.grid(True, linestyle='--')
plt.show()

print(f"总降雨量: {np.sum(sample_rainfall):.2f} mm")
print(f"SimpleRunoffModule 总产流量: {np.sum(simple_runoff_results):.2f} mm")
print(f"SCSCurveNumberModule 总产流量: {np.sum(scs_runoff_results):.2f} mm")

### 对比分析

从图中和总产流量中可以看到：
- **响应模式**: `SCS`模块的响应更“剧烈”，一旦降雨超过初损，就会产生较大的径流。而`SimpleRunoff`模块的响应则更“平缓”，产流量取决于前期的土壤含水量，因此在模拟初期的产流量较小，随着土壤变湿，产流系数才逐渐增大。
- **状态记忆**: 在第二次降雨事件（`t=10`开始）时，`SimpleRunoff`由于在第一次降雨后土壤仍然湿润，所以其产流响应比第一次更快、更强。而`SCS`模块没有这种“记忆”，它对两次独立的降雨事件的响应是相同的。

这两种模块各有优劣，适用于不同的建模场景和数据可用情况。